In [1]:
#

In [2]:
import math
from dataclasses import dataclass
from typing import Optional, Literal, List, Tuple

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split

# --------------------------
# Labels (fixed 6 classes)
# --------------------------
LABEL2ID = {
    "None": 0,
    "Religious Hate": 1,
    "Sexism": 2,
    "Political Hate": 3,
    "Profane": 4,
    "Abusive": 5,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_CLASSES = 6


# --------------------------
# Utilities
# --------------------------
def parse_logits_str(s: str, num_classes: int = NUM_CLASSES) -> np.ndarray:
    """Convert '2.0 -1.0 ...' -> np.array([2.0, -1.0, ...], float32) length=num_classes."""
    v = np.fromstring(str(s).strip(), sep=" ", dtype=np.float32)
    if v.size != num_classes:
        raise ValueError(f"Expected {num_classes} logits, got {v.size} for value: {s!r}")
    return v


def load_ground_truth(ground_truth_path: str) -> pd.DataFrame:
    df = pd.read_csv(ground_truth_path, sep="\t")

    # Normalize any variant names
    for i in range(1, 1000):  # support up to logits_999
        if f"logit_{i}" in df.columns and f"logits_{i}" not in df.columns:
            df = df.rename(columns={f"logit_{i}": f"logits_{i}"})

    logit_cols = [c for c in df.columns if c.startswith("logits_")]
    if not logit_cols:
        raise ValueError(f"No logits columns found in {ground_truth_path}!")

    required = {"id", "label_id"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    df["label_id"] = df["label_id"].astype("int64")

    # Sanity check
    for col in logit_cols:
        bad = df[col].isna().sum()
        if bad:
            print(f"Warning: {bad} rows have NaN in {col}")
        _ = parse_logits_str(df[col].iloc[0])  # quick check

    print(f"Loaded {len(df)} rows with {len(logit_cols)} logits columns.")
    return df


# --------------------------
# Dataset
# --------------------------
class NLogitsMulticlassDataset(Dataset):
    """
    Returns (list of z tensors, y)
      - z_list: list of [num_classes] tensors
      - y: int64 scalar
    """
    def __init__(self, df: pd.DataFrame, num_classes: int = NUM_CLASSES):
        self.df = df.reset_index(drop=True)
        self.logit_cols = [c for c in df.columns if c.startswith("logits_")]
        self.num_classes = num_classes

    def __len__(self): return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        z_list = [torch.from_numpy(parse_logits_str(row[c], self.num_classes)) for c in self.logit_cols]
        y = torch.tensor(int(row["label_id"]), dtype=torch.long)
        return z_list, y


# --------------------------
# Combiners
# --------------------------
class LinearCombinerN(nn.Module):
    """
    out[k] = b[k] + sum_m W[m,k] * z_m[k]
    """
    def __init__(self, num_models: int, num_classes: int = NUM_CLASSES):
        super().__init__()
        self.W = nn.Parameter(torch.full((num_models, num_classes), 1.0 / num_models))
        self.b = nn.Parameter(torch.zeros(num_classes))

    def forward(self, *z_list):  # each [B,6]
        Z = torch.stack(z_list, dim=1)  # [B, M, C]
        return (Z * self.W).sum(dim=1) + self.b


class MLPCombinerN(nn.Module):
    """
    Input: concat [z1|z2|...|zM] (M*C) → hidden → logits (C)
    """
    def __init__(self, num_models: int, num_classes: int = NUM_CLASSES, hidden: int = 128, p_drop: float = 0.1):
        super().__init__()
        in_dim = num_models * num_classes
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(p_drop),
            nn.Linear(hidden, num_classes),
        )
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, a=math.sqrt(5))
                nn.init.zeros_(m.bias)

    def forward(self, *z_list):
        x = torch.cat(z_list, dim=-1)  # [B, M*C]
        return self.net(x)


# --------------------------
# Training
# --------------------------
@dataclass
class TrainConfig:
    batch_size: int = 128
    epochs: int = 15
    lr: float = 2e-3
    weight_decay: float = 1e-4
    combine: Literal["linear", "mlp"] = "linear"
    mlp_hidden: int = 128
    dropout: float = 0.1
    temperature: Optional[float] = None
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    val_fraction: float = 0.15
    es_patience: int = 4
    num_workers: int = 0


@torch.no_grad()
def evaluate(model, loader, device) -> Tuple[float, float]:
    model.eval()
    crit = nn.CrossEntropyLoss()
    total_loss, total_correct, n = 0.0, 0.0, 0
    for z_list, y in loader:
        z_list = [z.to(device) for z in z_list]
        y = y.to(device)
        logits = model(*z_list)
        loss = crit(logits, y)
        total_loss += loss.item() * y.size(0)
        total_correct += (logits.argmax(-1) == y).sum().item()
        n += y.size(0)
    return total_loss / max(1, n), total_correct / max(1, n)


def train(df: pd.DataFrame, cfg: TrainConfig):
    ds = NLogitsMulticlassDataset(df)
    num_models = len(ds.logit_cols)

    n_total = len(ds)
    n_val = max(1, int(round(cfg.val_fraction * n_total)))
    n_train = n_total - n_val
    train_ds, val_ds = random_split(ds, [n_train, n_val], generator=torch.Generator().manual_seed(42))

    pin = (cfg.device == "cuda")
    collate = lambda batch: ([torch.stack([z[i] for z, _ in batch]) for i in range(num_models)],
                             torch.stack([y for _, y in batch]))

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                              collate_fn=collate, pin_memory=pin, num_workers=cfg.num_workers)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                            collate_fn=collate, pin_memory=pin, num_workers=cfg.num_workers)

    # Model
    if cfg.combine == "linear":
        base_model = LinearCombinerN(num_models, NUM_CLASSES)
    else:
        base_model = MLPCombinerN(num_models, NUM_CLASSES, hidden=cfg.mlp_hidden, p_drop=cfg.dropout)

    model = base_model.to(cfg.device)

    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    crit = nn.CrossEntropyLoss()

    best_val_loss, best_state, wait = float("inf"), None, 0

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        for z_list, y in train_loader:
            z_list = [z.to(cfg.device) for z in z_list]
            y = y.to(cfg.device)
            opt.zero_grad(set_to_none=True)
            logits = model(*z_list)
            loss = crit(logits, y)
            loss.backward()
            opt.step()

        val_loss, val_acc = evaluate(model, val_loader, cfg.device)
        print(f"Epoch {epoch:02d} | val_loss={val_loss:.4f} | val_acc={val_acc*100:.2f}%")

        if val_loss < best_val_loss - 1e-4:
            best_val_loss, best_state, wait = val_loss, {k: v.cpu().clone() for k, v in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= cfg.es_patience:
                print("Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model


# --------------------------
# Usage
# --------------------------
GROUND_TRUTH_PATH = "/kaggle/input/ensemble-v4/merged_logits_ensemble_val.tsv"

df = load_ground_truth(GROUND_TRUTH_PATH)
print(df.head())
cfg = TrainConfig(combine="mlp", epochs=15, val_fraction=0.35, es_patience=5)
model = train(df, cfg)

# Example prediction
ds = NLogitsMulticlassDataset(df)
loader = DataLoader(ds, batch_size=8, shuffle=False,
                    collate_fn=lambda batch: ([torch.stack([z[i] for z, _ in batch]) 
                                               for i in range(len(ds.logit_cols))],
                                              torch.stack([y for _, y in batch])))
z_list, y = next(iter(loader))
logits = model(*[z.to(cfg.device) for z in z_list])
preds = logits.argmax(dim=-1).cpu().tolist()
print("Sample predictions:", [ID2LABEL[p] for p in preds])


Loaded 7841 rows with 2 logits columns.
       id           label  label_id  \
0  166449  Political Hate         3   
1  267692         Abusive         5   
2  184031             NaN         0   
3  939131         Abusive         5   
4  210284         Abusive         5   

                                            logits_1  \
0  1.775258653759658 -0.7920978587127655 -0.91368...   
1  1.0409601108014597 -0.79790528076759 -1.165835...   
2  0.3542574346507664 -1.1257319714496277 -1.3662...   
3  1.2671743259349624 -1.0814771603778384 -0.8632...   
4  1.9107450893323021 -0.7662519637884508 -0.6316...   

                                            logits_2  
0  1.5532223039562776 -0.8432905801776686 -0.8650...  
1  0.057823113799300545 -0.8666657814543879 -0.72...  
2  0.9428362862902129 -1.1327054700018788 -0.9163...  
3  1.7179887315071734 -0.7580972708888032 -0.6607...  
4  0.8209106354104396 -0.6737691949185406 -0.9395...  
Epoch 01 | val_loss=0.7022 | val_acc=72.52%
Epoch 02 | val

In [3]:
# Predict on attached TSV and write submission.tsv (no training)
# Works with N logits columns: id, (logit_1|logits_1), (logit_2|logits_2), ..., (logit_N|logits_N)

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# --------------------------
# Labels (fixed 6 classes)
# --------------------------
LABEL2ID = {
    "None": 0,
    "Religious Hate": 1,
    "Sexism": 2,
    "Political Hate": 3,
    "Profane": 4,
    "Abusive": 5,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_CLASSES = 6

INPUT_TSV = "/kaggle/input/ensemble-5/merged_logits_ensemble_pred.tsv"
OUTPUT_TSV = "submission.tsv"

# --------------------------
# Utilities
# --------------------------
def parse_logits_str(s: str, num_classes: int = NUM_CLASSES) -> np.ndarray:
    v = np.fromstring(str(s).strip(), sep=" ", dtype=np.float32)
    if v.size != num_classes:
        raise ValueError(f"Expected {num_classes} logits, got {v.size} for value: {s!r}")
    return v

def load_for_inference(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t")

    # Normalize names: logit_i -> logits_i
    for i in range(1, 2000):  # generous cap
        k_old, k_new = f"logit_{i}", f"logits_{i}"
        if k_old in df.columns and k_new not in df.columns:
            df = df.rename(columns={k_old: k_new})

    logit_cols = [c for c in df.columns if c.startswith("logits_")]
    if not logit_cols:
        raise ValueError(f"{path} must contain at least one logits_* column")

    if "id" not in df.columns:
        raise ValueError("Missing required column: 'id'")

    # Friendly types
    try:
        df["id"] = df["id"].astype(int)
    except Exception:
        pass
    for c in logit_cols:
        df[c] = df[c].astype(str)

    return df

# --------------------------
# Dataset for N logits
# --------------------------
class NLogitsPredictDS(Dataset):
    def __init__(self, df: pd.DataFrame, num_classes: int = NUM_CLASSES):
        self.df = df.reset_index(drop=True)
        self.logit_cols = [c for c in df.columns if c.startswith("logits_")]
        self.num_classes = num_classes

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        z_list = [torch.from_numpy(parse_logits_str(row[c], self.num_classes)) for c in self.logit_cols]  # each [C]
        rid = int(row["id"])
        return z_list, rid

def make_collate_fn(num_models: int):
    def _collate(batch):
        # batch: list of (z_list, id)
        # Stack per-model across batch -> list of tensors of shape [B, C]
        z_stacked = []
        for m in range(num_models):
            z_m = torch.stack([sample[0][m] for sample in batch], dim=0)  # [B, C]
            z_stacked.append(z_m)
        ids = [sample[1] for sample in batch]
        return z_stacked, ids
    return _collate

# --------------------------
# Minimal equal-weight combiner for N models
# --------------------------
class EqualWeightCombinerN(nn.Module):
    def forward(self, *z_list):  # each [B, C]
        # Average logits across models
        Z = torch.stack(z_list, dim=0)  # [M, B, C]
        return Z.mean(dim=0)            # [B, C]

# --------------------------
# Prediction
# --------------------------
df_pred = load_for_inference(INPUT_TSV)
logit_cols = [c for c in df_pred.columns if c.startswith("logits_")]
num_models = len(logit_cols)

ds = NLogitsPredictDS(df_pred)
loader = DataLoader(ds, batch_size=512, shuffle=False, collate_fn=make_collate_fn(num_models))

# Use trained _model & _cfg.device if present; otherwise fallback to equal-weight N-averaging
try:
    model = _model  # noqa: F821
    device = _cfg.device  # noqa: F821
except NameError:
    model = EqualWeightCombinerN()
    device = "cuda" if torch.cuda.is_available() else "cpu"

model = model.to(device)
model.eval()

all_ids, all_preds = [], []
with torch.no_grad():
    for z_list, rid in loader:
        z_list = [z.to(device) for z in z_list]      # each [B, C]
        logits = model(*z_list)                       # [B, C]
        preds = logits.argmax(dim=-1).cpu().tolist()  # [B]
        all_ids.extend(rid)
        all_preds.extend(preds)

labels = [ID2LABEL[int(i)] for i in all_preds]
submission = pd.DataFrame({
    "id": all_ids,
    "label": labels,
    "model_name": "custom",  # change if desired
})
submission.to_csv(OUTPUT_TSV, sep="\t", index=False)
print(f"Wrote predictions to {OUTPUT_TSV} (used {num_models} logits columns)")


Wrote predictions to submission.tsv (used 2 logits columns)


In [4]:
# # predict_with_trained_model.py
# import pandas as pd
# import numpy as np
# import torch
# from torch.utils.data import DataLoader

# # --- import your combiner classes and helpers from the training script ---
# from train_multiclass_ensemble_from_tsv import (
#     merge_pred_by_id, ThreeLogitsPredictDataset,
#     LinearCombiner, MLPCombiner, PredictConfig, load_combiner,
#     predict_from_tsvs
# )


# # Paths to your prediction TSVs
# ensemble_paths = [
#     "/kaggle/input/merge-prediction/subtask_1A_pred_1.tsv",
#     "/kaggle/input/merge-prediction/subtask_1A_pred_2.tsv",
#     "/kaggle/input/merge-prediction/subtask_1A_pred_3.tsv",
# ]

# # Path to your trained model checkpoint
# ckpt_path = "combiner.pt"   # <-- make sure this file exists (saved during training)

# # Configuration (must match the combiner type you trained)
# cfg = PredictConfig(
#     combine="linear",   # or "mlp", depending on what you trained
#     mlp_hidden=128,
#     dropout=0.1,
#     temperature=None,
#     batch_size=512,
# )

# # Run predictions
# preds = predict_from_tsvs(
#     ensemble_paths=ensemble_paths,
#     model_ckpt_path=ckpt_path,
#     cfg=cfg,
#     out_path="predictions.tsv",   # saves results
#     return_probs=True,
# )

# print(preds.head())
